# Count-Min Sketch: Frequency Estimation

## Learning Objectives

1. **Explain** the Count-Min sketch and why it only over-estimates
2. **Derive** the error bound as a function of sketch dimensions
3. **Implement** Count-Min and evaluate point query accuracy
4. **Apply** Count-Min to heavy hitters detection

## Problem: Item Frequency in a Stream

Given a stream over a universe $[m]$ (e.g., URLs, words, user IDs), answer:
- **Point query:** what is the frequency $f_x$ of item $x$?
- **Heavy hitters:** which items have $f_i \geq \phi n$?

**Exact:** maintain a frequency table of size $m$ — too large if $m$ is huge.

**Goal:** estimate $f_x$ with small additive error using $O(1/\epsilon \cdot \log(1/\delta))$ space.

## Count-Min Sketch Structure

Maintain a $d \times w$ counter array $C$ where:
- $w = \lceil e/\epsilon \rceil$ (columns)
- $d = \lceil \ln(1/\delta) \rceil$ (rows)
- Each row has its own hash function $h_j : U \to [w]$

**Insert item $x$:** for each row $j$, increment $C[j][h_j(x)]$

**Query $f_x$:** return $\hat{f}_x = \min_j C[j][h_j(x)]$

**Guarantee:** with probability $1-\delta$:
$$f_x \leq \hat{f}_x \leq f_x + \epsilon n$$

Only over-estimates; never under-estimates.

In [ ]:
import math, hashlib, random
from collections import Counter

class CountMinSketch:
    def __init__(self, epsilon=0.01, delta=0.01):
        self.w = math.ceil(math.e / epsilon)
        self.d = math.ceil(math.log(1/delta))
        self.C = [[0]*self.w for _ in range(self.d)]
        self.n = 0
        # Random hash seeds
        rng = random.Random(42)
        self.seeds = [rng.randint(0, 2**32) for _ in range(self.d)]
        print(f"CountMinSketch: epsilon={epsilon}, delta={delta}")
        print(f"  d={self.d} rows, w={self.w} columns, total cells={self.d*self.w}")

    def _hash(self, item, row):
        key = f"{self.seeds[row]}:{item}".encode()
        return int(hashlib.md5(key).hexdigest(),16) % self.w

    def add(self, item, count=1):
        self.n += count
        for j in range(self.d):
            self.C[j][self._hash(item, j)] += count

    def query(self, item):
        return min(self.C[j][self._hash(item, j)] for j in range(self.d))

# Test
rng = random.Random(7)
# Zipfian-ish distribution
items = ['item_%d' % i for i in range(100)]
weights = [1/i for i in range(1,101)]
total = sum(weights)
weights = [w/total for w in weights]

stream = rng.choices(items, weights=weights, k=100000)
true_freq = Counter(stream)

cms = CountMinSketch(epsilon=0.005, delta=0.01)
for x in stream: cms.add(x)

# Evaluate accuracy on top-20 items
top20 = true_freq.most_common(20)
print(f"
{'Item':>8} | {'True freq':>10} | {'CMS est':>10} | {'Over-est':>10}")
for item, true_f in top20[:10]:
    est = cms.query(item)
    print(f"{item:>8} | {true_f:>10} | {est:>10} | {est-true_f:>+10}")

In [ ]:
# Heavy hitters using CMS
phi = 0.01  # threshold: items with frequency >= phi*n
threshold = phi * cms.n

print(f"
Heavy hitters (freq >= {phi:.1%} = {threshold:.0f} occurrences):")
# Scan once to find candidates (require tracking items seen)
candidates = set(stream)  # In practice, maintain a separate summary
for item in sorted(candidates, key=lambda x: cms.query(x), reverse=True)[:10]:
    est = cms.query(item)
    true_f = true_freq[item]
    if est >= threshold:
        tag = "TRUE" if true_f >= threshold else "FALSE POS"
        print(f"  {item:>8}: est={est:>6}, true={true_f:>6}  [{tag}]")

## Comparison with Other Sketches

| Sketch | Query type | Error | Space |
|--------|-----------|-------|-------|
| Count-Min | Point frequency | Additive $\epsilon n$ | $O(\frac{1}{\epsilon} \log \frac{1}{\delta})$ |
| AMS | $F_2$ (sum of squares) | Multiplicative $(1\pm\epsilon)$ | $O(\frac{1}{\epsilon^2} \log \frac{1}{\delta})$ |
| Flajolet-Martin | Distinct count $F_0$ | Multiplicative | $O(\log m \log \frac{1}{\delta})$ |
| Bloom filter | Membership | One-sided FP | $O(n \log \frac{1}{\epsilon})$ bits |

Count-Min is widely used in production: Twitter trending, network traffic analysis, database query optimization.